In [1]:
import pandas as pd
data = pd.read_json(path_or_buf="/kaggle/input/datasets/borezz/data-relevance-of-organizations/archive/data_for_train.jsonl", lines=True)

In [2]:
data.head()

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized,relevance_new
0,Стриптиз клубы ижевск,"Удмуртская Республика, Ижевск, Береговая улица",More,Яхт-клуб,184649161205,"Яхт-клуб More предлагает прогулки на яхтах, ар...",0.0,Общий обзор отзывов: яхт-клуб «More» в Ижевске...,0.0
1,Подшипник KAMAZ 45105000000000,"Ростовская область, Таганрог, улица Ломоносова...","Подшипник; Kompaniya Podshipnik; Подшипник, ТД...",Магазин автозапчастей и автотоваров,84348429597,None,0.1,Организация занимается продажей автозапчастей ...,0.1
2,такелажные компании,"Киров, улица Менделеева, 2",Кировская такелажная компания; Kirovskaya Take...,Строительные леса,1095265306,Кировская такелажная компания предлагает строп...,1.0,Организация занимается продажей строительных л...,1.0
3,баннеры,"Свердловская область, Екатеринбург, улица Карл...","Дабл, Онлайн-Полиграфия; Double Print; Студия ...",Полиграфические услуги,1221953140,None,1.0,Организация занимается полиграфическими услуга...,1.0
4,лучший ресторан москвы 2016,"Москва, улица Крымский Вал, 9с1",Сыроварня; Syrovarnya; Сыроварня в Парке Горьк...,Ресторан,191911335353,Ресторан предлагает разнообразные блюда: от за...,1.0,Организация занимается приготовлением и подаче...,1.0


In [3]:
data['relevance'].value_counts()

relevance
1.0    15455
0.0    14075
0.1     4564
Name: count, dtype: int64

# Бейзлайн

## Подготовка и метрика

In [4]:

import pandas as pd
import numpy as np

train = pd.read_json("/kaggle/input/datasets/borezz/data-relevance-of-organizations/archive/data_for_train.jsonl", lines=True)
eval_df = pd.read_json("/kaggle/input/datasets/borezz/data-relevance-of-organizations/archive/data_for_eval.jsonl", lines=True)

# Маппинг шкалы в классы
LABEL_MAP = {0.0: "IRRELEVANT", 0.1: "PARTIAL", 1.0: "RELEVANT"}
INV_MAP = {v: k for k, v in LABEL_MAP.items()}

train["label"] = train["relevance"].map(LABEL_MAP)

In [5]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(y_true, y_pred):
    return {
        "accuracy_3cls": accuracy_score(y_true, y_pred),
        "macro_f1_3cls": f1_score(y_true, y_pred, average="macro"),
        # бинаризация: PARTIAL -> к нерелевантным 
        "accuracy_bin": accuracy_score(
            [1 if y == "RELEVANT" else 0 for y in y_true],
            [1 if y == "RELEVANT" else 0 for y in y_pred]),
    }

In [6]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, val_idx = next(gss.split(train, groups=train["Text"]))
train_part, val_part = train.iloc[tr_idx], train.iloc[val_idx]

In [7]:
print(train["Text"].nunique(), "уникальных запросов на", len(train), "строк")
print(train["Text"].value_counts().head(10))  

9251 уникальных запросов на 34094 строк
Text
фастфуд                    94
Шиномонтаж                 71
автосервисы шиномонтаж     41
шиномонтаж                 39
Обмен валюты               35
Автосервисы, шиномонтаж    34
мотель                     33
Банки                      32
банки                      28
Быстрое питание            26
Name: count, dtype: int64


In [8]:
overlap = set(eval_df["Text"]) & set(train["Text"])
print(f"Пересечение запросов eval и train: {len(overlap)} из {eval_df['Text'].nunique()}")

Пересечение запросов eval и train: 493 из 542


## Промпт для полей

In [10]:
SYSTEM_PROMPT = """Ты — опытный асессор Яндекс.Карт. Оцениваешь релевантность организации \
рубричному поисковому запросу. Пользователь ищет ТИП места (например, «ресторан с верандой», \
«стриптиз клубы»), а не конкретную организацию.

Шкала оценки:
- RELEVANT — организация относится к запрошенному типу заведений И удовлетворяет всем \
атрибутам запроса (атрибуты подтверждаются рубрикой, ценами/услугами или отзывами).
- PARTIAL — организация близка к запросу, но есть натяжка: смежный тип заведения \
(запрос «кофейня» → пекарня, где есть кофе), либо ключевой атрибут запроса не подтверждается \
данными, но и не противоречит им, либо запрошенная услуга второстепенна для организации.
- IRRELEVANT — тип заведения не совпадает с запросом, либо атрибут запроса явно отсутствует \
или противоречит данным.

Правила:
1. ТИП заведения важнее всего. Определяй его в первую очередь по рубрике и списку услуг. \
Если тип не совпадает — IRRELEVANT, даже если отзывы содержат подходящие слова. \
Пример: запрос «стриптиз клуб» и яхт-клуб с «романтичным отдыхом» в отзывах — IRRELEVANT.
2. Город/регион в запросе сверяй с адресом организации. Другой город — IRRELEVANT.
3. Атрибуты запроса (веранда, живая музыка, детская комната, круглосуточно) ищи в услугах и \
отзывах. Явное подтверждение → RELEVANT; нет упоминаний → PARTIAL; противоречие → IRRELEVANT.
4. Не додумывай факты, которых нет в данных.

Отвечай строго в JSON:
{"entity_type_match": "да/нет/частично", "attributes_check": "<проверка атрибутов запроса>", \
"reasoning": "<итоговое обоснование, 1-2 предложения>", "label": "RELEVANT" | "PARTIAL" | "IRRELEVANT"}"""


def build_user_prompt(row) -> str:
    return f"""Запрос: {row['Text']}

Организация:
- Название: {row['name']}
- Основная рубрика: {row['normalized_main_rubric_name_ru']}
- Адрес: {row['address']}
- Услуги и цены: {str(row.get('prices_summarized', ''))[:1500] or '—'}
- Сводка отзывов: {str(row.get('reviews_summarized', ''))[:2500] or '—'}"""

## Инференс с zero-shot

In [12]:
import json, re

VALID = {"RELEVANT", "PARTIAL", "IRRELEVANT"}

def parse_label(text: str):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            label = json.loads(m.group(0)).get("label", "").upper()
            if label in VALID:
                return label, text
    except json.JSONDecodeError:
        pass
    # fallback, ищем метку в тексте
    for lab in ["IRRELEVANT", "PARTIAL", "RELEVANT"]:  # IRRELEVANT раньше, т.к. содержит "RELEVANT"
        if lab in text.upper():
            return lab, text
    return "IRRELEVANT", text  

In [13]:
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map="auto")


def predict_row_hf(row, fewshot_rows=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if fewshot_rows is not None:
        for _, ex in fewshot_rows.iterrows():
            messages.append({"role": "user", "content": build_user_prompt(ex)})
            messages.append({"role": "assistant", "content": json.dumps(
                {"reasoning": "…", "label": ex["label"]}, ensure_ascii=False)})
    messages.append({"role": "user", "content": build_user_prompt(row)})

    # chat template превращает messages в один текст с спецтокенами
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    answer = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_label(answer)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
import os, gc, hashlib
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-large"

WORK_DIR = "/kaggle/working/cache"                 
INPUT_DIR = "/kaggle/input/datasets/my-emb-cache"           

def row_key(row):
    return f"query: {row['Text']} | рубрика: {row['normalized_main_rubric_name_ru']} | {row['name']}"

def get_embeddings(df, cache_name="train_emb", model_name=MODEL_NAME):
    texts = [row_key(r) for _, r in df.iterrows()]
    fingerprint = hashlib.md5(
        (model_name + "||" + "\n".join(texts)).encode()
    ).hexdigest()

    for base in (INPUT_DIR, WORK_DIR):
        npy_path = os.path.join(base, f"{cache_name}.npy")
        fp_path  = os.path.join(base, f"{cache_name}.fingerprint")
        if os.path.exists(npy_path) and os.path.exists(fp_path):
            if open(fp_path).read() == fingerprint:
                print(f"Загружаю эмбеддинги из кэша: {npy_path}")
                return np.load(npy_path)
            else:
                print(f"Кэш в {base} устарел (данные/модель изменились)")

    print("Кэш не найден — считаю эмбеддинги")
    gc.collect(); torch.cuda.empty_cache()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = 128            
    if device == "cuda":
        model.half()                      

    emb = model.encode(texts, normalize_embeddings=True,
                       batch_size=32, show_progress_bar=True).astype(np.float32)

    del model; gc.collect(); torch.cuda.empty_cache()

    os.makedirs(WORK_DIR, exist_ok=True)
    np.save(os.path.join(WORK_DIR, f"{cache_name}.npy"), emb)
    with open(os.path.join(WORK_DIR, f"{cache_name}.fingerprint"), "w") as f:
        f.write(fingerprint)
    print(f"Кэш сохранён: {WORK_DIR}/{cache_name}.npy")
    return emb


core = train_part
core_emb = get_embeddings(core, cache_name="train_emb")

Кэш не найден — считаю эмбеддинги


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/906 [00:00<?, ?it/s]

Кэш сохранён: /kaggle/working/cache/train_emb.npy


In [17]:
val_part.head()

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized,relevance_new,label
1,Подшипник KAMAZ 45105000000000,"Ростовская область, Таганрог, улица Ломоносова...","Подшипник; Kompaniya Podshipnik; Подшипник, ТД...",Магазин автозапчастей и автотоваров,84348429597,None,0.1,Организация занимается продажей автозапчастей ...,0.1,PARTIAL
2,такелажные компании,"Киров, улица Менделеева, 2",Кировская такелажная компания; Kirovskaya Take...,Строительные леса,1095265306,Кировская такелажная компания предлагает строп...,1.0,Организация занимается продажей строительных л...,1.0,RELEVANT
6,архитектура бровей,"Москва, Зеленодольская улица, 45к1",M_Studio; Студия Маникюра,Ногтевая студия,119071783820,M_Studio предлагает широкий спектр услуг по ух...,1.0,"Организация занимается маникюром, педикюром и ...",1.0,RELEVANT
33,пивной ресторан ленинградское шоссе,"Москва, Туристская улица, 13, корп. 2",BierЛога; BierLoga; Бирлога; Bier Лога; Пивной...,Ресторан,1397146248,Ресторан и паб BierЛога предлагает разнообразн...,1.0,Организация занимается ресторанным обслуживани...,1.0,RELEVANT
37,агентство по подбору персонала октябрь россия,"Нижегородская область, Арзамас, улица 9 Мая, 2А",Бюро по подбору персонала АМЗ; Бюро по подбору...,"Кадровые агентства , вакансии",167321085705,None,0.0,Организация занимается подбором персонала. Отз...,0.0,IRRELEVANT


In [18]:
from tqdm import tqdm

def run(df, fewshot_fn=None):
    preds, raws = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        fs = fewshot_fn(row) if fewshot_fn else None
        label, raw = predict_row_hf(row, fs)
        preds.append(label); raws.append(raw)
    return preds, raws

val_sample = val_part.sample(400, random_state=42)
preds, raws = run(val_sample)                      # zero-shot
print(evaluate(val_sample["label"].tolist(), preds))

100%|██████████| 400/400 [1:17:47<00:00, 11.67s/it]

{'accuracy_3cls': 0.4575, 'macro_f1_3cls': 0.33772778398591846, 'accuracy_bin': 0.595}


In [19]:
eval_df["label"] = eval_df["relevance_new"].map(LABEL_MAP)

preds, raws = run(eval_df)                      # весь eval, 500 строк
print(evaluate(eval_df["label"].tolist(), preds))

100%|██████████| 570/570 [1:49:45<00:00, 11.55s/it]

{'accuracy_3cls': 0.4, 'macro_f1_3cls': 0.30733169037750646, 'accuracy_bin': 0.48947368421052634}
